# 周末报价策略回测分析

## 策略说明
本回测分析JPYCNH在周末（周六00:00~06:00北京时间）的报价策略，
研究是否可以根据USDCNH的价格变动来调整JPYCNH的报价。

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 设置绘图风格
plt.style.use('seaborn-v0_8-whitegrid')

# 项目路径
sys.path.insert(0, r"c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system")
OUTPUT_DIR = r"c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system\output"

print("环境配置完成!")

## 1. 数据加载

In [ ]:
def load_data():
    """加载JPYCNH和USDCNH数据"""
    # JPYCNH - 修正后的数据
    jpycnh_file = os.path.join(OUTPUT_DIR, "JPYCNH_Curncy_bidask_1min_corrected_20260130_115938.xlsx")
    
    # USDCNH - 原始数据
    usdcnh_file = os.path.join(OUTPUT_DIR, "USDCNH_Curncy_bidask_1s_20260116_144224.xlsx")
    
    print(f"正在加载 JPYCNH: {os.path.basename(jpycnh_file)}")
    df_jpycnh = pd.read_excel(jpycnh_file)
    df_jpycnh['timestamp'] = pd.to_datetime(df_jpycnh['timestamp'])
    df_jpycnh.set_index('timestamp', inplace=True)
    
    print(f"正在加载 USDCNH: {os.path.basename(usdcnh_file)}")
    df_usdcnh = pd.read_excel(usdcnh_file)
    df_usdcnh['timestamp'] = pd.to_datetime(df_usdcnh['timestamp'])
    df_usdcnh.set_index('timestamp', inplace=True)
    
    print(f"\nJPYCNH 数据量: {df_jpycnh.shape}")
    print(f"USDCNH 数据量: {df_usdcnh.shape}")
    
    return df_jpycnh, df_usdcnh

df_jpycnh, df_usdcnh = load_data()

## 2. 周末数据分析

In [ ]:
def analyze_weekend_data(df_jpycnh, df_usdcnh):
    """分析周末数据特征"""
    # 添加星期列
    df_jpycnh['weekday'] = df_jpycnh.index.dayofweek
    df_usdcnh['weekday'] = df_usdcnh.index.dayofweek
    
    # 筛选周六数据 (北京时间 00:00~06:00)
    jpycnh_sat = df_jpycnh[df_jpycnh['weekday'] == 5].copy()
    usdcnh_sat = df_usdcnh[df_usdcnh['weekday'] == 5].copy()
    
    print(f"JPYCNH 周六数据行数: {len(jpycnh_sat)}")
    print(f"USDCNH 周六数据行数: {len(usdcnh_sat)}")
    
    # 按小时分布
    print("\nJPYCNH 周六每小时数据分布:")
    jpycnh_sat['hour'] = jpycnh_sat.index.hour
    hour_counts = jpycnh_sat.groupby('hour').size()
    for hour, count in hour_counts.items():
        print(f"  {hour:02d}:00 - {count} 行")
    
    return jpycnh_sat, usdcnh_sat

jpycnh_sat, usdcnh_sat = analyze_weekend_data(df_jpycnh.copy(), df_usdcnh.copy())

## 3. 计算周末交易指标

In [ ]:
def calculate_weekend_metrics(df_jpycnh, df_usdcnh):
    """计算周末交易指标"""
    # 添加日期和星期
    df_jpycnh['weekday'] = df_jpycnh.index.dayofweek
    df_jpycnh['date'] = df_jpycnh.index.date
    df_usdcnh['weekday'] = df_usdcnh.index.dayofweek
    df_usdcnh['date'] = df_usdcnh.index.date
    
    # 获取周六日期
    jpycnh_sat_dates = df_jpycnh[df_jpycnh['weekday'] == 5]['date'].unique()
    usdcnh_sat_dates = df_usdcnh[df_usdcnh['weekday'] == 5]['date'].unique()
    
    common_dates = sorted(set(jpycnh_sat_dates) & set(usdcnh_sat_dates))
    print(f"共同的周六日期数量: {len(common_dates)}")
    
    results = []
    
    for sat_date in common_dates:
        # 周六数据 (00:00~06:00)
        jpycnh_sat = df_jpycnh[(df_jpycnh['date'] == sat_date) & (df_jpycnh.index.hour < 6)]
        usdcnh_sat = df_usdcnh[(df_usdcnh['date'] == sat_date) & (df_usdcnh.index.hour < 6)]
        
        if jpycnh_sat.empty or usdcnh_sat.empty:
            continue
        
        # 计算指标
        jpycnh_open = jpycnh_sat['mid'].iloc[0] if 'mid' in jpycnh_sat.columns else (jpycnh_sat['bid'].iloc[0] + jpycnh_sat['ask'].iloc[0]) / 2
        jpycnh_close = jpycnh_sat['mid'].iloc[-1] if 'mid' in jpycnh_sat.columns else (jpycnh_sat['bid'].iloc[-1] + jpycnh_sat['ask'].iloc[-1]) / 2
        jpycnh_high = jpycnh_sat['mid'].max() if 'mid' in jpycnh_sat.columns else ((jpycnh_sat['bid'] + jpycnh_sat['ask']) / 2).max()
        jpycnh_low = jpycnh_sat['mid'].min() if 'mid' in jpycnh_sat.columns else ((jpycnh_sat['bid'] + jpycnh_sat['ask']) / 2).min()
        
        usdcnh_open = usdcnh_sat['mid'].iloc[0] if 'mid' in usdcnh_sat.columns else (usdcnh_sat['bid'].iloc[0] + usdcnh_sat['ask'].iloc[0]) / 2
        usdcnh_close = usdcnh_sat['mid'].iloc[-1] if 'mid' in usdcnh_sat.columns else (usdcnh_sat['bid'].iloc[-1] + usdcnh_sat['ask'].iloc[-1]) / 2
        usdcnh_high = usdcnh_sat['mid'].max() if 'mid' in usdcnh_sat.columns else ((usdcnh_sat['bid'] + usdcnh_sat['ask']) / 2).max()
        usdcnh_low = usdcnh_sat['mid'].min() if 'mid' in usdcnh_sat.columns else ((usdcnh_sat['bid'] + usdcnh_sat['ask']) / 2).min()
        
        # 计算收益率 (基点)
        jpycnh_return = (jpycnh_close - jpycnh_open) / jpycnh_open * 10000
        usdcnh_return = (usdcnh_close - usdcnh_open) / usdcnh_open * 10000
        
        # 波动范围 (基点)
        jpycnh_range = (jpycnh_high - jpycnh_low) / jpycnh_open * 10000
        usdcnh_range = (usdcnh_high - usdcnh_low) / usdcnh_open * 10000
        
        results.append({
            'date': sat_date,
            'jpycnh_open': jpycnh_open,
            'jpycnh_close': jpycnh_close,
            'jpycnh_return_bps': jpycnh_return,
            'jpycnh_range_bps': jpycnh_range,
            'usdcnh_open': usdcnh_open,
            'usdcnh_close': usdcnh_close,
            'usdcnh_return_bps': usdcnh_return,
            'usdcnh_range_bps': usdcnh_range,
        })
    
    df_results = pd.DataFrame(results)
    
    if not df_results.empty:
        corr = df_results['jpycnh_return_bps'].corr(df_results['usdcnh_return_bps'])
        
        print(f"\n周末交易统计 ({len(df_results)} 个周六)")
        print("-" * 50)
        print(f"\nJPYCNH 周末统计:")
        print(f"  平均收益: {df_results['jpycnh_return_bps'].mean():.2f} 基点")
        print(f"  收益标准差: {df_results['jpycnh_return_bps'].std():.2f} 基点")
        print(f"  平均波动范围: {df_results['jpycnh_range_bps'].mean():.2f} 基点")
        
        print(f"\nUSDCNH 周末统计:")
        print(f"  平均收益: {df_results['usdcnh_return_bps'].mean():.2f} 基点")
        print(f"  收益标准差: {df_results['usdcnh_return_bps'].std():.2f} 基点")
        print(f"  平均波动范围: {df_results['usdcnh_range_bps'].mean():.2f} 基点")
        
        print(f"\nJPYCNH vs USDCNH 收益相关性: {corr:.4f}")
    
    return df_results

metrics_df = calculate_weekend_metrics(df_jpycnh.copy(), df_usdcnh.copy())

## 4. 可视化: 周末收益分布

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. JPYCNH vs USDCNH 散点图
ax1 = axes[0, 0]
ax1.scatter(metrics_df['usdcnh_return_bps'], metrics_df['jpycnh_return_bps'], 
            alpha=0.7, s=80, c='steelblue', edgecolors='white', linewidth=0.5)
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
corr = metrics_df['jpycnh_return_bps'].corr(metrics_df['usdcnh_return_bps'])
ax1.set_xlabel('USDCNH 周末收益 (基点)', fontsize=12)
ax1.set_ylabel('JPYCNH 周末收益 (基点)', fontsize=12)
ax1.set_title(f'JPYCNH vs USDCNH 周末收益\n相关系数: {corr:.4f}', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 添加趋势线
z = np.polyfit(metrics_df['usdcnh_return_bps'], metrics_df['jpycnh_return_bps'], 1)
p = np.poly1d(z)
x_line = np.linspace(metrics_df['usdcnh_return_bps'].min(), metrics_df['usdcnh_return_bps'].max(), 100)
ax1.plot(x_line, p(x_line), 'r--', alpha=0.8, label=f'趋势线 (斜率={z[0]:.3f})')
ax1.legend()

# 2. JPYCNH 收益分布
ax2 = axes[0, 1]
ax2.hist(metrics_df['jpycnh_return_bps'], bins=20, color='coral', alpha=0.7, edgecolor='white')
ax2.axvline(x=metrics_df['jpycnh_return_bps'].mean(), color='red', linestyle='--', 
            label=f'均值: {metrics_df["jpycnh_return_bps"].mean():.2f}', linewidth=2)
ax2.axvline(x=0, color='black', linestyle='-', alpha=0.5)
ax2.set_xlabel('收益 (基点)', fontsize=12)
ax2.set_ylabel('频次', fontsize=12)
ax2.set_title('JPYCNH 周末收益分布', fontsize=14, fontweight='bold')
ax2.legend()

# 3. USDCNH 收益分布
ax3 = axes[1, 0]
ax3.hist(metrics_df['usdcnh_return_bps'], bins=20, color='lightseagreen', alpha=0.7, edgecolor='white')
ax3.axvline(x=metrics_df['usdcnh_return_bps'].mean(), color='darkgreen', linestyle='--', 
            label=f'均值: {metrics_df["usdcnh_return_bps"].mean():.2f}', linewidth=2)
ax3.axvline(x=0, color='black', linestyle='-', alpha=0.5)
ax3.set_xlabel('收益 (基点)', fontsize=12)
ax3.set_ylabel('频次', fontsize=12)
ax3.set_title('USDCNH 周末收益分布', fontsize=14, fontweight='bold')
ax3.legend()

# 4. 波动范围对比
ax4 = axes[1, 1]
x = np.arange(len(metrics_df))
width = 0.35
ax4.bar(x - width/2, metrics_df['jpycnh_range_bps'], width, label='JPYCNH', color='coral', alpha=0.8)
ax4.bar(x + width/2, metrics_df['usdcnh_range_bps'], width, label='USDCNH', color='lightseagreen', alpha=0.8)
ax4.set_xlabel('周六序号', fontsize=12)
ax4.set_ylabel('波动范围 (基点)', fontsize=12)
ax4.set_title('周末波动范围对比', fontsize=14, fontweight='bold')
ax4.legend()
ax4.axhline(y=metrics_df['jpycnh_range_bps'].mean(), color='coral', linestyle='--', alpha=0.5)
ax4.axhline(y=metrics_df['usdcnh_range_bps'].mean(), color='lightseagreen', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'weekend_returns_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存至: weekend_returns_analysis.png")

## 5. 回测报价策略

In [ ]:
def backtest_pricing_strategy(df_jpycnh, df_usdcnh, spread_bps=5.0):
    """
    回测周末报价策略
    
    策略:
    - 使用USDCNH变动作为参考
    - 根据USDCNH变化调整JPYCNH报价
    - 当客户与我们报价成交时赚取点差
    """
    # 将USDCNH重采样到1分钟
    df_usdcnh_1min = df_usdcnh.resample('1min').last().ffill()
    
    # 添加列
    df_jpycnh['weekday'] = df_jpycnh.index.dayofweek
    df_jpycnh['date'] = df_jpycnh.index.date
    df_usdcnh_1min['weekday'] = df_usdcnh_1min.index.dayofweek
    df_usdcnh_1min['date'] = df_usdcnh_1min.index.date
    
    # 计算中间价
    if 'mid' not in df_jpycnh.columns:
        df_jpycnh['mid'] = (df_jpycnh['bid'] + df_jpycnh['ask']) / 2
    if 'mid' not in df_usdcnh_1min.columns:
        df_usdcnh_1min['mid'] = (df_usdcnh_1min['bid'] + df_usdcnh_1min['ask']) / 2
    
    # 获取周六数据
    jpycnh_sat = df_jpycnh[(df_jpycnh['weekday'] == 5) & (df_jpycnh.index.hour < 6)].copy()
    usdcnh_sat = df_usdcnh_1min[(df_usdcnh_1min['weekday'] == 5) & (df_usdcnh_1min.index.hour < 6)].copy()
    
    # 共同时间戳
    common_idx = jpycnh_sat.index.intersection(usdcnh_sat.index)
    
    if len(common_idx) < 10:
        return None
    
    # 计算收益率
    jpycnh_aligned = jpycnh_sat.loc[common_idx].copy()
    usdcnh_aligned = usdcnh_sat.loc[common_idx].copy()
    
    jpycnh_aligned['jpycnh_ret'] = jpycnh_aligned['mid'].pct_change() * 10000
    usdcnh_aligned['usdcnh_ret'] = usdcnh_aligned['mid'].pct_change() * 10000
    
    spread_half = spread_bps / 2
    
    # 测试不同调整因子
    adjustment_factors = [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.75, 1.0, 1.25, 1.5]
    
    results = []
    cumulative_pnl = {}
    
    for adj_factor in adjustment_factors:
        pnl_total = 0
        trades = 0
        wins = 0
        pnl_series = [0]
        
        for i in range(1, len(common_idx)):
            ts = common_idx[i]
            
            usdcnh_move = usdcnh_aligned.loc[ts, 'usdcnh_ret']
            jpycnh_actual = jpycnh_aligned.loc[ts, 'jpycnh_ret']
            
            if pd.isna(usdcnh_move) or pd.isna(jpycnh_actual):
                pnl_series.append(pnl_series[-1])
                continue
            
            jpycnh_predicted = usdcnh_move * adj_factor
            quote_error = jpycnh_actual - jpycnh_predicted
            
            if abs(quote_error) < spread_half:
                pnl = spread_half
                wins += 1
            else:
                pnl = spread_half - abs(quote_error)
            
            pnl_total += pnl
            trades += 1
            pnl_series.append(pnl_total)
        
        avg_pnl = pnl_total / trades if trades > 0 else 0
        win_rate = wins / trades if trades > 0 else 0
        
        results.append({
            'spread_bps': spread_bps,
            'adj_factor': adj_factor,
            'trades': trades,
            'total_pnl_bps': pnl_total,
            'avg_pnl_bps': avg_pnl,
            'win_rate': win_rate,
        })
        cumulative_pnl[adj_factor] = pnl_series
    
    return pd.DataFrame(results), cumulative_pnl, common_idx

# 测试不同点差
spreads = [3.0, 5.0, 7.0, 10.0]
all_results = []
all_cumulative_pnl = {}

for spread in spreads:
    print(f"\n正在回测点差 = {spread} 基点...")
    result, cum_pnl, common_idx = backtest_pricing_strategy(df_jpycnh.copy(), df_usdcnh.copy(), spread_bps=spread)
    if result is not None:
        all_results.append(result)
        all_cumulative_pnl[spread] = cum_pnl

all_results_df = pd.concat(all_results, ignore_index=True)
print("\n回测完成!")

## 6. 回测结果汇总

In [ ]:
# 创建结果表格
print("\n" + "="*80)
print("回测结果汇总")
print("="*80)

for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    best_idx = df_spread['total_pnl_bps'].idxmax()
    best = df_spread.loc[best_idx]
    
    print(f"\n点差 = {spread:.0f} 基点:")
    print(f"  最优调整因子: {best['adj_factor']:.2f}")
    print(f"  总收益: {best['total_pnl_bps']:.2f} 基点")
    print(f"  平均每笔收益: {best['avg_pnl_bps']:.4f} 基点")
    print(f"  胜率: {best['win_rate']*100:.1f}%")

# 显示详细表格
print("\n" + "="*80)
print("详细回测结果 (5基点点差)")
print("="*80)
df_5bps = all_results_df[all_results_df['spread_bps'] == 5.0].copy()
df_5bps['win_rate_pct'] = df_5bps['win_rate'] * 100
print(df_5bps[['adj_factor', 'trades', 'total_pnl_bps', 'avg_pnl_bps', 'win_rate_pct']].to_string(index=False))

## 7. 可视化: 回测结果

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'3.0': '#FF6B6B', '5.0': '#4ECDC4', '7.0': '#45B7D1', '10.0': '#96C93D'}

# 1. 不同点差下的总收益对比
ax1 = axes[0, 0]
for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    ax1.plot(df_spread['adj_factor'], df_spread['total_pnl_bps'], 
             marker='o', linewidth=2, markersize=6,
             label=f'{spread:.0f} 基点点差', color=colors[str(spread)])
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('调整因子', fontsize=12)
ax1.set_ylabel('总收益 (基点)', fontsize=12)
ax1.set_title('不同点差下的总收益', fontsize=14, fontweight='bold')
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)

# 2. 胜率对比
ax2 = axes[0, 1]
for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    ax2.plot(df_spread['adj_factor'], df_spread['win_rate'] * 100, 
             marker='s', linewidth=2, markersize=6,
             label=f'{spread:.0f} 基点点差', color=colors[str(spread)])
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50%基准')
ax2.set_xlabel('调整因子', fontsize=12)
ax2.set_ylabel('胜率 (%)', fontsize=12)
ax2.set_title('不同点差下的胜率', fontsize=14, fontweight='bold')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 105])

# 3. 累计收益曲线 (5基点点差)
ax3 = axes[1, 0]
cum_pnl_5bps = all_cumulative_pnl.get(5.0, {})
selected_factors = [0.0, 0.25, 0.5, 1.0]
for factor in selected_factors:
    if factor in cum_pnl_5bps:
        ax3.plot(cum_pnl_5bps[factor], linewidth=1.5, label=f'因子={factor}')
ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax3.set_xlabel('交易次数', fontsize=12)
ax3.set_ylabel('累计收益 (基点)', fontsize=12)
ax3.set_title('累计收益曲线 (5基点点差)', fontsize=14, fontweight='bold')
ax3.legend(loc='best')
ax3.grid(True, alpha=0.3)

# 4. 最优调整因子对比
ax4 = axes[1, 1]
best_factors = []
best_pnls = []
best_winrates = []

for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    best_idx = df_spread['total_pnl_bps'].idxmax()
    best = df_spread.loc[best_idx]
    best_factors.append(best['adj_factor'])
    best_pnls.append(best['total_pnl_bps'])
    best_winrates.append(best['win_rate'] * 100)

x = np.arange(len(spreads))
width = 0.35

bars1 = ax4.bar(x - width/2, best_factors, width, label='最优调整因子', color='steelblue', alpha=0.8)
ax4_twin = ax4.twinx()
bars2 = ax4_twin.bar(x + width/2, best_winrates, width, label='胜率 (%)', color='coral', alpha=0.8)

ax4.set_xlabel('点差 (基点)', fontsize=12)
ax4.set_ylabel('最优调整因子', fontsize=12, color='steelblue')
ax4_twin.set_ylabel('胜率 (%)', fontsize=12, color='coral')
ax4.set_xticks(x)
ax4.set_xticklabels([f'{s:.0f}' for s in spreads])
ax4.set_title('不同点差的最优参数', fontsize=14, fontweight='bold')

# 添加数值标签
for i, (f, w) in enumerate(zip(best_factors, best_winrates)):
    ax4.annotate(f'{f:.2f}', (x[i] - width/2, f + 0.02), ha='center', fontsize=10)
    ax4_twin.annotate(f'{w:.1f}%', (x[i] + width/2, w + 1), ha='center', fontsize=10)

ax4.legend(loc='upper left')
ax4_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'backtest_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存至: backtest_results.png")

## 8. 每小时分析

In [ ]:
def analyze_hourly_patterns(df_jpycnh, df_usdcnh):
    """分析每小时的相关性和波动"""
    # 重采样
    df_usdcnh_1min = df_usdcnh.resample('1min').last().ffill()
    
    df_jpycnh['weekday'] = df_jpycnh.index.dayofweek
    df_usdcnh_1min['weekday'] = df_usdcnh_1min.index.dayofweek
    
    if 'mid' not in df_jpycnh.columns:
        df_jpycnh['mid'] = (df_jpycnh['bid'] + df_jpycnh['ask']) / 2
    if 'mid' not in df_usdcnh_1min.columns:
        df_usdcnh_1min['mid'] = (df_usdcnh_1min['bid'] + df_usdcnh_1min['ask']) / 2
    
    # 周六数据
    jpycnh_sat = df_jpycnh[(df_jpycnh['weekday'] == 5) & (df_jpycnh.index.hour < 6)].copy()
    usdcnh_sat = df_usdcnh_1min[(df_usdcnh_1min['weekday'] == 5) & (df_usdcnh_1min.index.hour < 6)].copy()
    
    common_idx = jpycnh_sat.index.intersection(usdcnh_sat.index)
    
    jpycnh_aligned = jpycnh_sat.loc[common_idx].copy()
    usdcnh_aligned = usdcnh_sat.loc[common_idx].copy()
    
    jpycnh_aligned['jpycnh_ret'] = jpycnh_aligned['mid'].pct_change() * 10000
    usdcnh_aligned['usdcnh_ret'] = usdcnh_aligned['mid'].pct_change() * 10000
    
    jpycnh_aligned['hour'] = jpycnh_aligned.index.hour
    usdcnh_aligned['hour'] = usdcnh_aligned.index.hour
    
    hourly_stats = []
    for hour in range(6):
        mask = jpycnh_aligned['hour'] == hour
        if mask.sum() > 10:
            corr = jpycnh_aligned.loc[mask, 'jpycnh_ret'].corr(usdcnh_aligned.loc[mask, 'usdcnh_ret'])
            jpycnh_vol = jpycnh_aligned.loc[mask, 'jpycnh_ret'].std()
            usdcnh_vol = usdcnh_aligned.loc[mask, 'usdcnh_ret'].std()
            hourly_stats.append({
                'hour': hour,
                'correlation': corr,
                'jpycnh_vol': jpycnh_vol,
                'usdcnh_vol': usdcnh_vol,
                'data_points': mask.sum()
            })
    
    return pd.DataFrame(hourly_stats)

hourly_df = analyze_hourly_patterns(df_jpycnh.copy(), df_usdcnh.copy())

print("\n每小时分析 (北京时间):")
print(hourly_df.to_string(index=False))

In [ ]:
# 绘制每小时分析图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. 每小时相关性
ax1 = axes[0]
bars = ax1.bar(hourly_df['hour'], hourly_df['correlation'], color='steelblue', alpha=0.8, edgecolor='white')
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('小时 (北京时间)', fontsize=12)
ax1.set_ylabel('相关系数', fontsize=12)
ax1.set_title('JPYCNH vs USDCNH 每小时相关性', fontsize=14, fontweight='bold')
ax1.set_xticks(hourly_df['hour'])
ax1.set_xticklabels([f'{h:02d}:00' for h in hourly_df['hour']])

# 添加数值标签
for bar, corr in zip(bars, hourly_df['correlation']):
    height = bar.get_height()
    ax1.annotate(f'{corr:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

# 2. 每小时波动率
ax2 = axes[1]
x = np.arange(len(hourly_df))
width = 0.35
ax2.bar(x - width/2, hourly_df['jpycnh_vol'], width, label='JPYCNH', color='coral', alpha=0.8)
ax2.bar(x + width/2, hourly_df['usdcnh_vol'], width, label='USDCNH', color='lightseagreen', alpha=0.8)
ax2.set_xlabel('小时 (北京时间)', fontsize=12)
ax2.set_ylabel('波动率 (基点)', fontsize=12)
ax2.set_title('每小时波动率对比', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([f'{h:02d}:00' for h in hourly_df['hour']])
ax2.legend()

# 3. 数据点数量
ax3 = axes[2]
ax3.bar(hourly_df['hour'], hourly_df['data_points'], color='mediumpurple', alpha=0.8, edgecolor='white')
ax3.set_xlabel('小时 (北京时间)', fontsize=12)
ax3.set_ylabel('数据点数量', fontsize=12)
ax3.set_title('每小时数据点数量', fontsize=14, fontweight='bold')
ax3.set_xticks(hourly_df['hour'])
ax3.set_xticklabels([f'{h:02d}:00' for h in hourly_df['hour']])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'hourly_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\n图表已保存至: hourly_analysis.png")

## 9. 结论与建议

In [ ]:
print("\n" + "="*80)
print("结论与建议")
print("="*80)

# 找出最优参数
for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    best_idx = df_spread['total_pnl_bps'].idxmax()
    best = df_spread.loc[best_idx]
    
print("\n【主要发现】")
print("-" * 60)

# 相关性分析
overall_corr = metrics_df['jpycnh_return_bps'].corr(metrics_df['usdcnh_return_bps'])
print(f"\n1. JPYCNH与USDCNH周末收益相关性: {overall_corr:.4f}")
if abs(overall_corr) < 0.3:
    print("   → 相关性较低，USDCNH对JPYCNH的预测能力有限")
elif abs(overall_corr) < 0.6:
    print("   → 相关性中等，可适度参考USDCNH进行调整")
else:
    print("   → 相关性较高，可较多参考USDCNH进行调整")

print(f"\n2. 最优调整因子建议:")
for spread in spreads:
    df_spread = all_results_df[all_results_df['spread_bps'] == spread]
    best_idx = df_spread['total_pnl_bps'].idxmax()
    best = df_spread.loc[best_idx]
    print(f"   点差{spread:.0f}基点: 调整因子={best['adj_factor']:.2f}, 胜率={best['win_rate']*100:.1f}%")

print(f"\n3. 每小时相关性差异:")
best_hour = hourly_df.loc[hourly_df['correlation'].abs().idxmax()]
worst_hour = hourly_df.loc[hourly_df['correlation'].abs().idxmin()]
print(f"   最高相关性小时: {int(best_hour['hour']):02d}:00 (相关性={best_hour['correlation']:.4f})")
print(f"   最低相关性小时: {int(worst_hour['hour']):02d}:00 (相关性={worst_hour['correlation']:.4f})")

print("\n" + "="*80)
print("【策略建议】")
print("="*80)
print("""
1. 由于JPYCNH与USDCNH的周末相关性较低，建议采用较小的调整因子
   (0.25-0.50左右)，避免过度依赖USDCNH的价格变动。

2. 对于较大的点差（7-10基点），可以适当提高调整因子，
   因为点差提供了更大的保护空间。

3. 建议根据不同时段的相关性差异，动态调整报价策略：
   - 相关性较高的时段：可以更多参考USDCNH
   - 相关性较低的时段：建议保持更保守的报价

4. 建议持续监控和更新相关性数据，以优化报价策略。
""")

print("\n回测分析完成!")

---
## 附录: 保存结果

In [ ]:
# 保存回测结果到Excel
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = os.path.join(OUTPUT_DIR, f'backtest_full_results_{timestamp}.xlsx')

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 周末指标
    metrics_df.to_excel(writer, sheet_name='周末指标', index=False)
    
    # 回测结果
    all_results_df.to_excel(writer, sheet_name='回测结果', index=False)
    
    # 每小时分析
    hourly_df.to_excel(writer, sheet_name='每小时分析', index=False)

print(f"\n结果已保存至: {output_file}")